In [16]:
#load the sentiment sample
import pandas as pd

sentiment_sample = pd.read_csv(
    "../../data/samples/yelp_sentiment_sample.csv"
)

print("Shape:", sentiment_sample.shape)
print(sentiment_sample.head())

Shape: (2193, 11)
                review_id                 user_id             business_id  \
0  t-8mpC0ryIc-cdnwoF8Hsw  5WTVFPWRpHvGVMUmUjtxJg  -361Hc0tlxSYdrH_C3OgzA   
1  rjnyXstBa7SUtlfrf-moyA  d2Qobgu7myzpEuEd6jTB9w  -361Hc0tlxSYdrH_C3OgzA   
2  h46h5_2USejVTkfM3byl7g  B1CvITVmO9aD6NFk0U_Y1A  -361Hc0tlxSYdrH_C3OgzA   
3  jQHWfUlgObQRh40PUDYU_w  OPg_YrZvFY6fFvwSgR8ekw  -361Hc0tlxSYdrH_C3OgzA   
4  _YFKHUDdzhgMhKS0m7oDSw  4v_1I47f3MEUXHkibt6CLA  -AWclhh1_2VnqPylPgBU3g   

   stars  useful  funny  cool  \
0      5       1      0     1   
1      1       0      0     0   
2      5      11      1     8   
3      4       0      0     0   
4      1       0      0     0   

                                                text                 date  \
0  A HIDDEN GEM!!!!!!!!! I had the Chelo Kabob Ba...  2021-06-16 01:38:46   
1  We had been to this restaurant before and it w...  2021-05-09 15:32:40   
2  I am half-Iranian and absolutely LOVE Persian ...  2021-04-28 01:21:29   
3  The place

In [17]:
# Dataset Integrity and Review Length Analysis

null_ids = sentiment_sample["review_id"].isna().sum()
null_texts = sentiment_sample["text"].isna().sum()
dup_ids = sentiment_sample["review_id"].duplicated().sum()

print("=== DATASET INTEGRITY CHECK ===")
print(f"Total reviews in sample: {len(sentiment_sample)}")
print(f"Null review_ids: {null_ids}")
print(f"Null review texts: {null_texts}")
print(f"Duplicate review_ids: {dup_ids}")


# Calculate review lengths

sentiment_sample["char_len"] = (
    sentiment_sample["text"]
    .fillna("")
    .astype(str)
    .str.len()
)

sentiment_sample["word_len"] = (
    sentiment_sample["text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)


print("\n=== REVIEW LENGTH DISTRIBUTION ===")

print(
    f"Character Lengths -> "
    f"Min: {sentiment_sample['char_len'].min()} | "
    f"Mean: {int(sentiment_sample['char_len'].mean())} | "
    f"Median: {int(sentiment_sample['char_len'].median())} | "
    f"Max: {sentiment_sample['char_len'].max()}"
)

print(
    f"Word Counts -> "
    f"Min: {sentiment_sample['word_len'].min()} | "
    f"Mean: {int(sentiment_sample['word_len'].mean())} | "
    f"Median: {int(sentiment_sample['word_len'].median())} | "
    f"Max: {sentiment_sample['word_len'].max()}"
)


# Percentile distribution

percentiles = [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

print("\n=== WORD COUNT PERCENTILES ===")

print(
    sentiment_sample["word_len"]
    .quantile(percentiles)
    .to_string()
)


# Preview the two longest reviews

print("\n=== LONGEST REVIEW PREVIEW ===")

top_longest = sentiment_sample.nlargest(2, "char_len")

for _, row in top_longest.iterrows():

    print(
        f"\nID: {row['review_id']} | "
        f"Characters: {row['char_len']} | "
        f"Words: {row['word_len']}"
    )

    print(f"Excerpt: {row['text'][:250]}...")

=== DATASET INTEGRITY CHECK ===
Total reviews in sample: 2193
Null review_ids: 0
Null review texts: 0
Duplicate review_ids: 0

=== REVIEW LENGTH DISTRIBUTION ===
Character Lengths -> Min: 51 | Mean: 471 | Median: 342 | Max: 4943
Word Counts -> Min: 8 | Mean: 86 | Median: 62 | Max: 894

=== WORD COUNT PERCENTILES ===
0.25     35.00
0.50     62.00
0.75    112.00
0.90    179.00
0.95    228.40
0.99    367.48

=== LONGEST REVIEW PREVIEW ===

ID: u4ZBMDyZGu4DopdOWrE3DA | Characters: 4943 | Words: 894
Excerpt: My family and I went here to get a nice breakfast after an early morning walk. It's become almost a weekly ritual where we would go out and try a new breakfast spot every week and finally came about to this place. 
Having lived in Los Angeles for hal...

ID: 8ytzZqAtfFHbZCwP9APdWQ | Characters: 4049 | Words: 756
Excerpt: Not one to do this kinda feedback but hopefully they may change something for a better service experience. This is meant to impact future customers who read the reviews

In [18]:
#define the compression function
import re

def compress_review(text, max_words=120):
    """
    Compress long reviews while preserving the beginning
    and ending of the review.
    """

    words = text.split()

    # Keep reviews that are already short
    if len(words) <= max_words:
        return text

    # Split long reviews into sentences
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?]) +", text)
        if sentence.strip()
    ]

    # If there are only a few sentences,
    # keep the first max_words words
    if len(sentences) <= 3:
        return " ".join(words[:max_words]) + "..."

    # Otherwise keep the first two and last two sentences
    return (
        " ".join(sentences[:2])
        + " ... "
        + " ".join(sentences[-2:])
    )

In [19]:
# Inspect compression on the 5 longest reviews

top_long = sentiment_sample.nlargest(5, "char_len")

for _, row in top_long.iterrows():

    original = row["text"]

    compressed = compress_review(
        original,
        max_words=120
    )

    print(
        f'=== REVIEW ID: {row["review_id"]} '
        f'({row["word_len"]} words) ==='
    )

    print(f"ORIGINAL:\n{original[:300]}...\n")

    print(f"COMPRESSED:\n{compressed}\n")

    print("-" * 80)

=== REVIEW ID: u4ZBMDyZGu4DopdOWrE3DA (894 words) ===
ORIGINAL:
My family and I went here to get a nice breakfast after an early morning walk. It's become almost a weekly ritual where we would go out and try a new breakfast spot every week and finally came about to this place. 
Having lived in Los Angeles for half my life, we were ecstatic to see California/Mexi...

COMPRESSED:
My family and I went here to get a nice breakfast after an early morning walk. It's become almost a weekly ritual where we would go out and try a new breakfast spot every week and finally came about to this place. ... From what I've read from other reviews perhaps other items on the menu taste better, but from my experience alone and if I'm comparing it to past experiences with both Mexican food and breakfast food, I can't recommend for much more than a try-once for this place. For some people I guess some of the food is good if you can stomach the price tag on some items, and I really wish eventually in the fut

In [20]:
import importlib
import sys
import pandas as pd

# Add src folder to system path
sys.path.append("../../src")

# Import and reload custom GenAI utility
import genai_utils

importlib.reload(genai_utils)
from genai_utils import analyze_reviews, get_groq_client

print("✓ Updated genai_utils successfully reloaded.")

✓ Updated genai_utils successfully reloaded.


In [21]:
# Verify GROQ_API_KEY environment variable and test connection
client = get_groq_client()
print("✓ Groq client initialized successfully.")

✓ Groq client initialized successfully.


In [22]:
import os
import requests

api_key = os.getenv("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"
headers = {"Authorization": f"Bearer {api_key}"}

response = requests.get(url, headers=headers)
if response.status_code == 200:
  models = [m["id"] for m in response.json()["data"]]
  print("✓ Available Groq Models:")
  for m in sorted(models):
    print(f"  • {m}")
else:
  print(f"Error fetching models: {response.status_code} - {response.text}")

✓ Available Groq Models:
  • allam-2-7b
  • canopylabs/orpheus-arabic-saudi
  • canopylabs/orpheus-v1-english
  • groq/compound
  • groq/compound-mini
  • meta-llama/llama-prompt-guard-2-22m
  • meta-llama/llama-prompt-guard-2-86m
  • openai/gpt-oss-120b
  • openai/gpt-oss-20b
  • openai/gpt-oss-safeguard-20b
  • qwen/qwen3.6-27b
  • qwen/qwen3.8-27b
  • whisper-large-v3
  • whisper-large-v3-turbo


In [23]:
# Quick test on diverse edge-case reviews to verify model & validation logic
df_test = pd.DataFrame([
    {"review_id": "test_pos", "text": "Amazing food and excellent service!"},
    {
        "review_id": "test_neg",
        "text": "Terrible experience, cold food and rude service.",
    },
    {
        "review_id": "test_mix",
        "text": "The food was great, but the wait time was ridiculous.",
    },
    {"review_id": "test_neu", "text": "Ordered the chicken sandwich for lunch."},
    {"review_id": "test_short", "text": "Good."},
])

df_test_results = analyze_reviews(
    df_reviews=df_test,
    text_col="text",
    batch_size=5,
    checkpoint_file="../../data/raw/yelp/test_checkpoint.csv",
    model="openai/gpt-oss-20b",
)

# Clean up test checkpoint file after sanity run
if os.path.exists("../../data/raw/yelp/test_checkpoint.csv"):
  os.remove("../../data/raw/yelp/test_checkpoint.csv")

# Safety assertions
assert len(df_test_results) == len(df_test), "Input/Output count mismatch!"
assert df_test_results["review_id"].nunique() == len(
    df_test
), "Duplicate review IDs found!"
assert set(df_test_results["review_id"]) == set(
    df_test["review_id"]
), "Missing review IDs!"
assert (
    df_test_results["sentiment_label"]
    .isin(["positive", "neutral", "negative"])
    .all()
), "Invalid label found!"
assert (
    (df_test_results["sentiment_score"] >= -1.0).all()
    and (df_test_results["sentiment_score"] <= 1.0).all()
), "Scores out of bounds!"

print("✓ All 5 sanity test assertions passed!")
display(df_test_results)

2026-09-02 16:00:55,452 - INFO - Processing remaining 5 reviews in 1 batches...
2026-09-02 16:00:58,013 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-09-02 16:00:58,015 - INFO - Batch 1/1 complete (5/5 total saved).


✓ All 5 sanity test assertions passed!


,review_id,sentiment_label,sentiment_score,sentiment_reason
0,test_pos,positive,0.90,Positive language praising food and service.
1,test_neg,negative,-0.85,Negative words describing cold food and rude s...
2,test_mix,neutral,0.00,"Mixed positive food, negative wait time."
3,test_neu,neutral,0.00,"No sentiment expressed, just statement."
4,test_short,positive,0.70,Short positive word indicates satisfaction.


In [24]:
# Load the 2,193 sentiment sample CSV dataset 
sample_path = "../../data/samples/yelp_sentiment_sample.csv"
df_sample = pd.read_csv(sample_path)

print(f"Loaded sentiment sample: {len(df_sample)} reviews.")
display(df_sample.head(2))

Loaded sentiment sample: 2193 reviews.


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,period,merchant_status
0,t-8mpC0ryIc-cdnwoF8Hsw,5WTVFPWRpHvGVMUmUjtxJg,-361Hc0tlxSYdrH_C3OgzA,5,1,0,1,A HIDDEN GEM!!!!!!!!! I had the Chelo Kabob Ba...,2021-06-16 01:38:46,earlier,Growing
1,rjnyXstBa7SUtlfrf-moyA,d2Qobgu7myzpEuEd6jTB9w,-361Hc0tlxSYdrH_C3OgzA,1,0,0,0,We had been to this restaurant before and it w...,2021-05-09 15:32:40,earlier,Growing


In [26]:
# Resume full execution 
df_review_sentiment = analyze_reviews(
    df_reviews=df_sample,
    text_col="compressed_text",
    batch_size=2,
    delay_between_batches=2,
    checkpoint_file="../../data/raw/yelp/yelp_review_sentiment_checkpoint.csv",
    model="openai/gpt-oss-20b",
)

print(f"✓ Complete! Successfully processed all {len(df_review_sentiment)} reviews.")

2026-09-02 16:01:56,180 - INFO - Column 'compressed_text' not found. Auto-compressing 'text'...
2026-09-02 16:01:56,210 - INFO - Found existing checkpoint file! Resuming with 2193 reviews already completed.
2026-09-02 16:01:56,222 - INFO - All reviews have already been processed in the checkpoint file!


✓ Complete! Successfully processed all 2193 reviews.


In [27]:
#read existing checkpoint
import os
import pandas as pd

checkpoint_path = "../../data/raw/yelp/yelp_review_sentiment_checkpoint.csv"
df_ckpt = pd.read_csv(checkpoint_path)
print(f"Initial checkpoint rows: {len(df_ckpt)}")

#filter out testing rows
df_cleaned = df_ckpt[
    ~df_ckpt["review_id"].astype(str).str.startswith("test_")
].copy()
print(f"Rows after dropping test records: {len(df_cleaned)}")

#filter out fallback rows
df_cleaned = df_cleaned[
    ~df_cleaned["sentiment_reason"]
    .astype(str)
    .str.contains("Fallback assigned", case=False, na=False)
].copy()
print(f"Rows after dropping fallbacks: {len(df_cleaned)}")

#overwrite checkpoint file
df_cleaned.to_csv(checkpoint_path, index=False)
print("✓ Checkpoint file cleanly overwritten.")


Initial checkpoint rows: 2193
Rows after dropping test records: 2193
Rows after dropping fallbacks: 2193
✓ Checkpoint file cleanly overwritten.


In [28]:
#verify output data frame integrity
print(f"Final Shape: {df_review_sentiment.shape}")
print(df_review_sentiment["sentiment_label"].value_counts(dropna=False))
assert (
    df_review_sentiment["sentiment_reason"].str.contains("Fallback").sum()
    == 0
), "Fallback rows present!"

Final Shape: (2193, 4)
sentiment_label
positive    1455
negative     627
neutral      111
Name: count, dtype: int64


#### GenAI Sentiment Output

The completed extraction contains **2,193 review-level records** with four fields:

| Field              | Description                          |
| ------------------ | ------------------------------------ |
| `review_id`        | Unique Yelp review identifier        |
| `sentiment_label`  | `positive`, `neutral`, or `negative` |
| `sentiment_score`  | Sentiment polarity from **-1 to 1**  |
| `sentiment_reason` | Concise explanation of the sentiment |

**Final sentiment distribution:** Positive: **1,455** · Negative: **627** · Neutral: **111**

Source metadata such as `business_id`, `stars`, `period`, and `merchant_status` remains available in `yelp_sentiment_sample.csv` and can be restored using `review_id`.

**Evidence-Based Prompting**

The prompt instructed the model to base sentiment classifications and explanations **only on information explicitly stated in the review** and not infer or invent unsupported details. For example, a negative review was not interpreted as a price complaint unless pricing or cost was explicitly mentioned.


In [29]:
#Save clean production sentiment file
from pathlib import Path
import pandas as pd

# Existing completed checkpoint
checkpoint_path = Path("../../data/raw/yelp/yelp_review_sentiment_checkpoint.csv")

# Stable production file in the existing samples directory
final_path = Path("../../data/samples/yelp_review_sentiment_final.csv")

# Load the completed checkpoint
df_sentiment_final = pd.read_csv(checkpoint_path)

print(f"Checkpoint rows: {len(df_sentiment_final):,}")
print(f"Checkpoint shape: {df_sentiment_final.shape}")
print(f"Columns: {df_sentiment_final.columns.tolist()}")

# Production integrity checks
assert len(df_sentiment_final) == 2193, \
    "Expected exactly 2,193 sentiment records."

assert df_sentiment_final["review_id"].nunique() == 2193, \
    "Duplicate review IDs found."

assert set(df_sentiment_final["sentiment_label"].dropna().unique()).issubset(
    {"positive", "neutral", "negative"}
), "Unexpected sentiment label found."

assert df_sentiment_final["sentiment_score"].between(-1, 1).all(), \
    "Sentiment score outside [-1, 1] detected."

assert not df_sentiment_final["sentiment_reason"].astype(str).str.contains(
    "Fallback",
    case=False,
    na=False
).any(), "Fallback rows detected."

# Save a separate copy as the stable production dataset
df_sentiment_final.to_csv(final_path, index=False)

print("\n✓ Production sentiment file created")
print(f"Path: {final_path}")
print(f"Rows: {len(df_sentiment_final):,}")
print(f"Shape: {df_sentiment_final.shape}")

Checkpoint rows: 2,193
Checkpoint shape: (2193, 4)
Columns: ['review_id', 'sentiment_label', 'sentiment_score', 'sentiment_reason']

✓ Production sentiment file created
Path: ..\..\data\samples\yelp_review_sentiment_final.csv
Rows: 2,193
Shape: (2193, 4)


In [30]:
# Verify the production file
df_production = pd.read_csv(final_path)

print("Production file verification")
print("-" * 40)
print(f"Shape: {df_production.shape}")
print(f"Columns: {df_production.columns.tolist()}")
print(f"Unique review IDs: {df_production['review_id'].nunique():,}")

print("\nSentiment distribution:")
print(df_production["sentiment_label"].value_counts())

Production file verification
----------------------------------------
Shape: (2193, 4)
Columns: ['review_id', 'sentiment_label', 'sentiment_score', 'sentiment_reason']
Unique review IDs: 2,193

Sentiment distribution:
sentiment_label
positive    1455
negative     627
neutral      111
Name: count, dtype: int64
